# TDC Native Tools - Modernização de Atendimento B2B

Case: reduzir tempo e aumentar qualidade de respostas técnicas para RFP/RFI em contas enterprise.


## 1. Instalação

Instale só as bibliotecas usadas no fluxo.


In [1]:
# Se necessário, descomente:
!pip install -U openai python-dotenv langchain langchain-openai



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. .env e conexão

No `.env`, use: `OCI_REGION`, `OCI_PROJECT_OCID`, `OCI_GENAI_API_KEY`, `OCI_FILE_SEARCH_VECTOR_STORE_ID`, `OCI_MCP_SERVER_URL`, `OCI_MCP_ALLOWED_TOOLS`.


In [2]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path.cwd() / ".env")

OCI_REGION = os.getenv("OCI_REGION")
OCI_PROJECT_OCID = os.getenv("OCI_PROJECT_OCID")
OCI_GENAI_API_KEY = os.getenv("OCI_GENAI_API_KEY")
OCI_FILE_SEARCH_VECTOR_STORE_ID = os.getenv("OCI_FILE_SEARCH_VECTOR_STORE_ID")
OCI_MCP_SERVER_URL = os.getenv("OCI_MCP_SERVER_URL")
OCI_MCP_ALLOWED_TOOLS = [t.strip() for t in os.getenv("OCI_MCP_ALLOWED_TOOLS", "").split(",") if t.strip()]

base_url = f"https://inference.generativeai.{OCI_REGION}.oci.oraclecloud.com/openai/v1"

client = OpenAI(
    base_url=base_url,
    api_key=OCI_GENAI_API_KEY,
    project=OCI_PROJECT_OCID,
)

print("Cliente OCI conectado.")


Cliente OCI conectado.


## 3. Trace opcional

Ative `SHOW_TOOL_TRACE=True` para ver chamadas e retornos das tools.


In [3]:
SHOW_TOOL_TRACE = True

def _output_items(response):
    return list(getattr(response, "output", None) or [])

def _item_attr(item, name, default=None):
    if hasattr(item, name):
        return getattr(item, name)
    if isinstance(item, dict):
        return item.get(name, default)
    return default

def print_tool_trace(response, title="Tool trace"):
    if not SHOW_TOOL_TRACE:
        return
    calls = []
    for item in _output_items(response):
        dump = item.model_dump() if hasattr(item, "model_dump") else item
        item_type = str(dump.get("type", ""))
        if any(x in item_type for x in ["function_call", "mcp", "code_interpreter", "file_search", "tool"]):
            calls.append(dump)
    print(f"\n[{title}]")
    print(json.dumps(calls, indent=2, ensure_ascii=False) if calls else "Sem chamadas de tool.")


## 4. Contexto da oportunidade (Function Tool)

Use para buscar dados da oportunidade e personalizar a proposta.


In [4]:
pipeline = {
    "Conta Varejo Prime": {
        "segmento": "varejo B2B",
        "objetivo": "acelerar respostas tecnicas para RFP e RFI",
        "prazo_resposta_dias": 7,
        "integracoes": ["CRM", "ERP", "Service Desk"],
        "foco": ["velocidade", "consistencia", "governanca"],
    }
}

def buscar_oportunidade(nome_conta: str):
    return pipeline.get(nome_conta, {"mensagem": "oportunidade nao encontrada"})

function_tools = [{
    "type": "function",
    "name": "buscar_oportunidade",
    "description": "Busca contexto da oportunidade comercial.",
    "parameters": {
        "type": "object",
        "properties": {"nome_conta": {"type": "string"}},
        "required": ["nome_conta"],
    },
}]

resp = client.responses.create(
    model="openai.gpt-oss-120b",
    tools=function_tools,
    input="Busque contexto da Conta Varejo Prime para montar proposta B2B.",
)
print_tool_trace(resp, "Function - chamada")

tool_outputs = []
for item in _output_items(resp):
    if _item_attr(item, "type") == "function_call" and _item_attr(item, "name") == "buscar_oportunidade":
        args = json.loads(_item_attr(item, "arguments", "{}"))
        result = buscar_oportunidade(**args)
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": _item_attr(item, "call_id"),
            "output": json.dumps(result, ensure_ascii=False),
        })

final_context = client.responses.create(
    model="openai.gpt-oss-120b",
    tools=function_tools,
    previous_response_id=resp.id,
    input=tool_outputs,
)
print_tool_trace(final_context, "Function - retorno")
print(final_context.output_text)



[Function - chamada]
[
  {
    "arguments": "{\n  \"nome_conta\": \"Conta Varejo Prime\"\n}",
    "call_id": "call_019dac4c-9b9b-7943-aff4-0e8c47ce57a5",
    "name": "buscar_oportunidade",
    "type": "function_call",
    "id": "call_019dac4c-9b9b-7943-aff4-0e8c47ce57a5",
    "namespace": null,
    "status": "completed"
  }
]

[Function - retorno]
Sem chamadas de tool.
**Contexto da Conta Varejo Prime**

| Item | Detalhes |
|------|----------|
| **Segmento** | Varejo B2B |
| **Objetivo Principal** | Acelerar respostas técnicas para RFP (Request for Proposal) e RFI (Request for Information) |
| **Prazo Médio de Resposta** | 7 dias |
| **Integrações Necessárias** | • CRM <br>• ERP <br>• Service Desk |
| **Focos Estratégicos** | • Velocidade <br>• Consistência <br>• Governança |

---

### Estrutura da Proposta B2B

1. **Introdução**  
   • Apresentação da sua empresa e expertise em soluções B2B para o varejo.  
   • Resumo das necessidades da Varejo Prime (acelerar respostas técnicas, ga

## 5. Base interna de proposta (File Search)

Use documentos internos para garantir consistência e padrão de resposta.


In [5]:
docs_dir = Path("tdc_atendimento_docs")
docs_dir.mkdir(exist_ok=True)

(docs_dir / "playbook_rfp_rfi.txt").write_text(
    "Playbook RFP/RFI: entender contexto, mapear obrigatorios, propor arquitetura, plano e proximos passos.",
    encoding="utf-8",
)
(docs_dir / "matriz_requisitos_b2b.txt").write_text(
    "Obrigatorios: seguranca, auditoria, integracao, SLA e plano por fases.",
    encoding="utf-8",
)
(docs_dir / "pricing_guardrails_b2b.txt").write_text(
    "Guardrails: MVP em 6-8 semanas, ROI antes de expandir, evitar customizacao precoce.",
    encoding="utf-8",
)
(docs_dir / "template_resposta_executiva.txt").write_text(
    "Template: contexto, solucao, plano, investimento e proximos passos.",
    encoding="utf-8",
)

print("Arquivos do atendimento B2B:")
for p in sorted(docs_dir.glob("*.txt")):
    print("-", p.name)


Arquivos do atendimento B2B:
- matriz_requisitos_b2b.txt
- playbook_rfp_rfi.txt
- pricing_guardrails_b2b.txt
- template_resposta_executiva.txt


## 6. Indexar documentos internos

Use para enviar os arquivos do case para o vector store.


In [6]:
internal_ids = []

for doc_path in sorted(Path("tdc_atendimento_docs").glob("*.txt")):
    with doc_path.open("rb") as f:
        up = client.files.create(file=f, purpose="user_data")
    client.vector_stores.files.create(
        vector_store_id=OCI_FILE_SEARCH_VECTOR_STORE_ID,
        file_id=up.id,
    )
    internal_ids.append(up.id)
    print(f"Indexado: {doc_path.name} -> {up.id}")

print("\nTotal indexado:", len(internal_ids))


Indexado: matriz_requisitos_b2b.txt -> file-ord-3ef05474-1dbf-44ab-b465-e2dcda0df690
Indexado: playbook_rfp_rfi.txt -> file-ord-703c3a1c-ecdb-4a0e-80c6-2b354db238e6
Indexado: pricing_guardrails_b2b.txt -> file-ord-8a82983a-cec1-4f7e-8af2-2972ce56ae70
Indexado: template_resposta_executiva.txt -> file-ord-79074570-f715-4bf5-9468-8700371bb26a

Total indexado: 4


## 7. File Search aplicado ao atendimento

Use para gerar resposta técnica baseada somente no conteúdo interno.


In [7]:
resp_fs = client.responses.create(
    model="openai.gpt-oss-120b",
    tools=[{"type": "file_search", "vector_store_ids": [OCI_FILE_SEARCH_VECTOR_STORE_ID]}],
    input=(
        "Com base apenas nos documentos internos, monte uma estrutura de resposta para RFP/RFI "
        "de atendimento B2B com foco em governanca e velocidade."
    ),
)
print_tool_trace(resp_fs, "File Search - chamada e retorno")
print(resp_fs.output_text)



[File Search - chamada e retorno]
[
  {
    "id": "call_019dac4c-c36d-7cd1-ace8-709c065fdb19",
    "queries": [
      "estrutura resposta RFP RFI B2B governança velocidade"
    ],
    "status": "completed",
    "type": "file_search_call",
    "results": null
  }
]
**Estrutura‑modelo de resposta a RFP / RFI B2B – Governança + Velocidade**  
*(Baseado apenas nos documentos internos: Playbook RFP/RFI, Template 30‑60‑90 dias e Protocolos Indústria Regulada)*  

---  

### 1. Apresentação Executiva  
| Item | Conteúdo (extraído do **Playbook RFP/RFI**) | objetivo |
|------|--------------------------------------------|----------|
| 1.1 Visão geral da solução | Resumo da proposta de arquitetura, indicadores de sucesso e próximos passos. | Demonstrar entendimento do problema e a proposta de valor. |
| 1.2 Diferenciais competitivos | Velocidade de implantação (30/60/90 dias) + modelo de governança robusto (audit trail, segregação de perfis, rollout controlado). | Evidenciar como a combinação d

## 8. MCP para benchmark de práticas públicas

Use MCP para trazer referências públicas de engenharia de proposta e documentação técnica.


In [8]:
mcp_tool = {
    "type": "mcp",
    "server_label": "deepwiki",
    "server_description": "Referencias publicas para benchmark de abordagem tecnica.",
    "server_url": OCI_MCP_SERVER_URL,
    "require_approval": "never",
}
if OCI_MCP_ALLOWED_TOOLS:
    mcp_tool["allowed_tools"] = OCI_MCP_ALLOWED_TOOLS

resp_mcp = client.responses.create(
    model="openai.gpt-oss-120b",
    tools=[mcp_tool],
    input=(
        "Busque referencias publicas de boas praticas para respostas tecnicas de projetos enterprise "
        "com integracao e governanca, e resuma os pontos que melhoram uma proposta B2B."
    ),
)
print_tool_trace(resp_mcp, "MCP - chamada e retorno")
print(resp_mcp.output_text)



[MCP - chamada e retorno]
[
  {
    "id": "mcpl_bfe8dd1162f44f0f9373c6607f293cf8",
    "server_label": "deepwiki",
    "tools": [
      {
        "input_schema": {
          "type": "object",
          "properties": {
            "repoName": {
              "type": "string"
            }
          },
          "required": [
            "repoName"
          ]
        },
        "name": "read_wiki_structure",
        "annotations": null,
        "description": "Get a list of documentation topics for a GitHub repository.\n\nArgs:\n    repoName: GitHub repository in owner/repo format (e.g. \"facebook/react\")"
      },
      {
        "input_schema": {
          "type": "object",
          "properties": {
            "repoName": {
              "type": "string"
            }
          },
          "required": [
            "repoName"
          ]
        },
        "name": "read_wiki_contents",
        "annotations": null,
        "description": "View documentation about a GitHub repositor

## 9. Code Interpreter para estimativa de esforço

Use para transformar requisitos em esforço e priorização objetiva.


In [9]:
base_text = resp_fs.output_text + "\n\n" + resp_mcp.output_text

resp_code = client.responses.create(
    model="xai.grok-code-fast-1",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    tool_choice="required",
    input=(
        "Com base no texto abaixo, monte uma tabela com 5 frentes (discovery, arquitetura, integracao, "
        "treinamento, operacao assistida). Para cada frente, estime esforco em semanas e prioridade (1-5). "
        "Depois sugira um plano de execucao de 8 semanas.\n\n"
        f"Texto base:\n{base_text}"
    ),
)
print_tool_trace(resp_code, "Code Interpreter - chamada e retorno")
print(resp_code.output_text)



[Code Interpreter - chamada e retorno]
[
  {
    "id": "ci_dfa2c38b-d6bf-28a5-949c-dc15a47904db_0",
    "code": "print(\"Análise do texto para extrair frentes de projeto baseadas no plano 30/60/90.\")\n\n# Frentes identificadas:\n# 1. Discovery (30 dias): Workshops, mapeamento de dados/integrações, definição de KPIs.\n# 2. Arquitetura: Proposta de arquitetura (passo 2 do Playbook), incluindo camadas de ingestão, processamento, etc.\n# 3. Integração: Parte do piloto (60 dias): Configuração sandbox, execução de casos de uso, medição de KPIs.\n# 4. Treinamento: Parte do 90 dias: Transferência de conhecimento (treinamento de equipe cliente).\n# 5. Operação Assistida: Parte do 90 dias: Roll-out controlado, monitoramento contínuo, relatórios de governança.\n\n# Estimativas:\n# Total plano: 90 dias (~12 semanas), mas usuário pediu 8 semanas, então ajustar proporcionalmente.\n# Esforço: Somar a algo próximo de 8 semanas.\n# Prioridade: Sequencial, 1 mais alta.\n\nfrentes = [\n    {\"frente\":

## 10. Agente final com LangChain

Use para consolidar contexto, evidências e estimativas em uma proposta executiva final.


In [10]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

VERBOSE_AGENT = True
MCP_REPOS = [
    "oracle-samples/db-sample-schemas",
    "oracle-samples/oracle-db-examples",
    "kubernetes/kubernetes",
]


def _log_tool(name: str, query: str, output: str):
    if not VERBOSE_AGENT:
        return
    print(f"\n[tool:{name}] input:\n{query}")
    print(f"[tool:{name}] output:\n{output[:1200]}\n")


@tool
def b2b_contexto(query: str) -> str:
    """Retorna dados da oportunidade comercial."""
    out = json.dumps(buscar_oportunidade("Conta Varejo Prime"), ensure_ascii=False)
    _log_tool("b2b_contexto", query, out)
    return out


@tool
def b2b_documentos(query: str) -> str:
    """Consulta base interna de proposta via File Search."""
    r = client.responses.create(
        model="openai.gpt-oss-120b",
        tools=[{"type": "file_search", "vector_store_ids": [OCI_FILE_SEARCH_VECTOR_STORE_ID]}],
        input=query,
    )
    print_tool_trace(r, "Agent -> File Search")
    _log_tool("b2b_documentos", query, r.output_text)
    return r.output_text


@tool
def b2b_benchmark(query: str) -> str:
    """Consulta benchmark externo via MCP com repos válidos owner/repo."""
    prompt = (
        "Use APENAS ferramentas MCP para responder. "
        f"Use ask_question com repoName={MCP_REPOS}. "
        f"Pergunta: {query}"
    )
    r = client.responses.create(
        model="openai.gpt-oss-120b",
        tools=[mcp_tool],
        input=prompt,
    )

    invalid_repo = False
    for item in _output_items(r):
        dump = item.model_dump() if hasattr(item, "model_dump") else item
        if dump.get("type") == "mcp_call" and "Invalid repoName format" in str(dump.get("output", "")):
            invalid_repo = True
            break

    if invalid_repo:
        retry_prompt = (
            "Falha de formato de repo detectada. Tente novamente usando EXATAMENTE estes repos owner/repo: "
            f"{MCP_REPOS}. Use ask_question e resuma em portugues para proposta B2B."
        )
        r = client.responses.create(
            model="openai.gpt-oss-120b",
            tools=[mcp_tool],
            input=retry_prompt,
        )

    print_tool_trace(r, "Agent -> MCP")
    _log_tool("b2b_benchmark", query, r.output_text)
    return r.output_text


@tool
def b2b_estimativa(query: str) -> str:
    """Calcula estimativas com Code Interpreter."""
    r = client.responses.create(
        model="xai.grok-code-fast-1",
        tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
        tool_choice="required",
        input=query,
    )
    print_tool_trace(r, "Agent -> Code Interpreter")
    _log_tool("b2b_estimativa", query, r.output_text)
    return r.output_text


llm = ChatOpenAI(
    model="openai.gpt-oss-120b",
    api_key=OCI_GENAI_API_KEY,
    base_url=base_url,
    default_headers={"project": OCI_PROJECT_OCID},
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=[b2b_contexto, b2b_documentos, b2b_benchmark, b2b_estimativa],
    system_prompt=(
        "Voce e um arquiteto de solucoes da Oracle para atendimento B2B. "
        "Antes da resposta final, chame obrigatoriamente as tools nesta ordem: "
        "b2b_contexto, b2b_documentos, b2b_benchmark, b2b_estimativa. "
        "Se alguma falhar, registre no bloco de riscos."
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Monte proposta para Conta Varejo Prime em 6 blocos: "
                    "1) contexto da oportunidade, 2) solucao recomendada Oracle, 3) benchmark de boas praticas, "
                    "4) estimativa de esforco em 8 semanas, 5) proximos passos executivos, "
                    "6) evidencias das tools (1 linha por tool com o que foi consultado)."
                ),
            }
        ]
    }
)

messages = result.get("messages", []) if isinstance(result, dict) else []

if VERBOSE_AGENT:
    print("\n===== TRILHA DO AGENTE (mensagens) =====")
    for i, msg in enumerate(messages, start=1):
        msg_type = getattr(msg, "type", msg.__class__.__name__)
        print(f"\n[{i}] {msg_type}")
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", json.dumps(tool_calls, ensure_ascii=False, indent=2))
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            print(str(content)[:1500])
        else:
            print(str(content)[:1500])

final_text = messages[-1].content if messages else result

print("\nProposta final:\n")
print(final_text)


[tool:b2b_contexto] input:
Conta Varejo Prime oportunidade comercial contexto
[tool:b2b_contexto] output:
{"segmento": "varejo B2B", "objetivo": "acelerar respostas tecnicas para RFP e RFI", "prazo_resposta_dias": 7, "integracoes": ["CRM", "ERP", "Service Desk"], "foco": ["velocidade", "consistencia", "governanca"]}


[Agent -> File Search]
[
  {
    "id": "call_019dac4d-785f-7682-ba26-2904d955816f",
    "queries": [
      "modelo de proposta conta varejo prime"
    ],
    "status": "completed",
    "type": "file_search_call",
    "results": null
  }
]

[tool:b2b_documentos] input:
proposta conta varejo prime documento modelo
[tool:b2b_documentos] output:
**PROPOSTA – CONTA VAREJO PRIME**  
*Modelo de documento*

---

**1. Capa**  

- Logotipo da empresa  
- Título: **Proposta Comercial – Conta Varejo Prime**  
- Cliente: **[Nome da empresa cliente]**  
- Data: **[dd/mm/aaaa]**  
- Versão: **1.0**  

---

**2. Sumário Executivo**  

| Item | Descrição |
|------|-----------|
| **Objeti